In [ ]:
# Rejith Asd reading and predicting code for old vasudha
import pandas as pd
import numpy as np
import joblib
import tkinter as tk
from tkinter import scrolledtext
import os
import struct
import time
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
from datetime import datetime

##Note:
##Keep all the models and inmput data (if nor connected to the ASAD Spectroradiometer) to the location named as  Model Folder###


# === ASD to CSV Conversion Class ===
data_type_dict = {
    0: "Raw", 1: "Reflectance", 2: "Radiance", 3: "No_Units",
    4: "Irradiance", 5: "QI", 6: "Transmittance", 7: "Unknown", 8: "Absorbance"
}
type_abbrev_dict = {"Raw": "Raw", "Reflectance": "Refl", "Reference": "Rfnc"}
data_format_dict = {0: "numeric", 1: "integer", 2: "double", 3: "Unknown"}
instrument_dict = {
    0: "Unknown", 1: "PSII", 2: "LSVNIR", 3: "FSVNIR", 4: "FSFR",
    5: "FSNIR", 6: "CHEM", 7: "FSFR Unattended"
}
flag1_dict = {1: "nir saturation", 2: "swir1 saturation", 3: "swir2 saturation", 8: "Tec1 alarm", 16: "Tec2 alarm"}

class ASDSpec:
    def __init__(self, asdfile, range_errors=True, data_format=None, do_normalize=True):
        self.filename = asdfile
        bref = open(asdfile, "rb")
        bref.seek(3)
        self.comments = bref.read(157).decode(errors='ignore').replace('\x00', "")
        bref.seek(182)
        seconds = struct.unpack('<L', bref.read(4))[0]
        self.aquisition_time = time.ctime(seconds)
        bref.seek(178)
        pv = struct.unpack("<B", bref.read(1))[0]
        self.program_version = "{}.{}".format(pv >> 4, pv & 7)
        fv = struct.unpack("<B", bref.read(1))[0]
        self.file_version = "{}.{}".format(fv >> 4, fv & 7)
        bref.seek(181)
        DC = struct.unpack("<B", bref.read(1))[0]
        self.vnir_dark_sub = (DC == 1)
        seconds = struct.unpack("<L", bref.read(4))[0]
        self.dark_meas_time = time.ctime(seconds)
        bref.seek(186)
        self.datatype = data_type_dict.get(struct.unpack("<B", bref.read(1))[0], "Unknown")
        seconds = struct.unpack("<L", bref.read(4))[0]
        self.white_meas_time = time.ctime(seconds)
        bref.seek(334)
        self.gps_trueHeading = struct.unpack("<d", bref.read(8))[0]
        self.gps_speed = struct.unpack("<d", bref.read(8))[0]
        self.gps_latitude = struct.unpack("<d", bref.read(8))[0]
        self.gps_longitude = struct.unpack("<d", bref.read(8))[0]
        self.gps_altitude = struct.unpack("<d", bref.read(8))[0]
        bref.seek(390)
        self.vnir_int_time = struct.unpack("<L", bref.read(4))[0]
        bref.seek(400)
        self.serial_number = str(struct.unpack("<h", bref.read(2))[0])
        bref.seek(204)
        self.num_channels = struct.unpack("<h", bref.read(2))[0]
        bref.seek(191)
        wstart = struct.unpack("<f", bref.read(4))[0]
        bref.seek(195)
        wstep = struct.unpack("<f", bref.read(4))[0]
        self.wavelength = [wstart + i * wstep for i in range(self.num_channels)]
        bref.seek(199)
        dataFormatCode = struct.unpack("<B", bref.read(1))[0]
        self.data_format = data_format_dict[dataFormatCode]
        struct_format = ["<{}f", "<{}l", "<{}d", None][dataFormatCode].format(self.num_channels)
        recbytes = [4, 4, 8, 0][dataFormatCode]
        bref.seek(418)
        self.inst_dynamic_range = 2 ** struct.unpack("<h", bref.read(2))[0]
        bref.seek(484)
        self.rawdata = struct.unpack(struct_format, bref.read(recbytes * self.num_channels))
        self.filesize = bref.seek(0, 2)
        if (484 + 2 * recbytes * self.num_channels) < self.filesize:
            bref.seek(484 + recbytes * self.num_channels)
            refbool, refsec, specsec = struct.unpack("<Hqq", bref.read(18))
            refdescsize = struct.unpack("<h", bref.read(2))[0]
            bref.read(refdescsize)
            self.refdata = list(struct.unpack(struct_format, bref.read(recbytes * self.num_channels)))
        else:
            self.refdata = None
        if do_normalize and self.datatype == "Raw":
            def normalize(wl, val):
                if wl <= 1000:
                    return val / self.vnir_int_time
                else:
                    return val
            self.rawdata = [normalize(self.wavelength[i], self.rawdata[i]) for i in range(self.num_channels)]

    def transform(self, output_type="Reflectance"):
        if output_type == "Reflectance":
            if self.refdata:
                return [self.rawdata[i] / self.refdata[i] for i in range(self.num_channels)]
            return list(self.rawdata)
        return list(self.rawdata)

    def to_csv(self, csvname, output_type="Reflectance", value_fmt=""):
        vals = self.transform(output_type)
        with open(csvname, 'w') as oref:
            oref.write("Wavelength,Reflectance\n")
            for i in range(len(self.wavelength)):
                oref.write(f"{self.wavelength[i]},{value_fmt.format(vals[i])}\n")

def process_asd_to_csv(input_file, output_file, output_type="Reflectance", sigdig=4):
    asdf = ASDSpec(input_file)
    valfmt = f"{{:.{sigdig}f}}"
    asdf.to_csv(output_file, output_type=output_type, value_fmt=valfmt)
    return output_file

# === Model Info ===
model_folder = "D:\\Tarun\\vasudha_python"
model_info = {
    "ASD_N_5nm_SVM.pkl": ("Av. Nitrogen", "Kg/hac"),
    "ASD_P_5nm_SVM.pkl": ("Av. Phosphorus", "Kg/hac"),
    "ASD_K_5nm_SVM.pkl": ("Av. Potassium", "Kg/hac"),
    "ASD_OC_5nm_SVM.pkl": ("Soil Organic Carbon", "%"),
    "ASD_pH_5nm_SVM.pkl": ("pH", ""),
    "ASD_EC_5nm_SVM.pkl": ("Electrical Conductivity", "dS/m"),
}

# === Main Loop: Processing ASD Files ===
asd_files = [f for f in os.listdir(model_folder) if f.lower().endswith(".asd")]

for asd_file in asd_files:
    input_path = os.path.join(model_folder, asd_file)
    output_csv = os.path.splitext(input_path)[0] + ".csv"

    try:
        # Step 1: Convert to CSV
        process_asd_to_csv(input_path, output_csv)
        asd_df = pd.read_csv(output_csv)
        spectral_values = asd_df.iloc[:, 1].values.tolist()

        # Step 2: Predict
        results = []
        result_dict = {}

        for model_file, (param_name, unit) in model_info.items():
            model_path = os.path.join(model_folder, model_file)
            if not os.path.exists(model_path):
                results.append(f"{param_name}: [Model not found]")
                result_dict[param_name] = "N/A"
                continue

            model_dict = joblib.load(model_path)
            model = model_dict["model"]
            features = model_dict["features"]

            if len(spectral_values) != len(features):
                results.append(f"{param_name}: [ERROR: Feature mismatch]")
                result_dict[param_name] = "Error"
                continue

            input_df = pd.DataFrame([spectral_values], columns=features)
            pred = model.predict(input_df)[0]
            results.append(f"{param_name}: {pred:.3f} {unit}")
            result_dict[param_name] = round(pred, 3)

               
           
        
        # Step 3: Save CSV
        output_result_csv = os.path.splitext(input_path)[0] + "_predicted_results.csv"
        pd.DataFrame.from_dict(result_dict, orient="index", columns=["Predicted Value"]).to_csv(output_result_csv)

        # Step 4: Show GUI
        root = tk.Tk()
        root.title(f"Prediction Results: {asd_file}")
        root.geometry("1000x600")

        left_frame = tk.Frame(root)
        left_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=10, pady=10)
        tk.Label(left_frame, text="Prediction using Vis-Near-Shortwave Infrared Spectroscopy", font=("Arial", 12, "bold")).pack()
        text_box = scrolledtext.ScrolledText(left_frame, width=40, height=25, font=("Arial", 11))
        text_box.pack(fill=tk.BOTH, expand=True)
        text_box.insert(tk.END, "\n".join(results))
        text_box.insert(tk.END, f"\n\nResults saved to:\n{output_result_csv}")
        text_box.config(state=tk.DISABLED)

        right_frame = tk.Frame(root)
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=10, pady=10)
        fig = Figure(figsize=(6, 5), dpi=100)
        ax = fig.add_subplot(111)
        ax.plot(asd_df.iloc[:, 0], asd_df.iloc[:, 1], color='green')
        ax.set_title('Reflectance Spectrum', fontsize=14)
        ax.set_xlabel('Wavelength (nm)')
        ax.set_ylabel('Reflectance')
        ax.grid(True)

        canvas = FigureCanvasTkAgg(fig, master=right_frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

        root.mainloop()

    finally:
        # Step 5: Cleanup
        os.remove(input_path)
      


In [ ]:
#!/usr/bin/env python3
# Rejith code batch
"""
Batch predict script (no GUI) for ASD/VNIR spectra.

Usage:
    python batch_predict_no_gui.py --input-dir /path/to/asd_files \
        --model-dir /path/to/models --output-dir ./predictions

Behavior:
- Tries to read spectrum files using specdal.Spectrum where available,
  otherwise falls back to a robust text parser that extracts numeric columns.
- Loads all *.pkl models from model-dir. Each .pkl may be:
    - A dict containing keys like 'pipeline' (sklearn Pipeline expecting features)
    - A dict containing 'model' and 'features' (list of column names or wavelength strings)
    - Or a plain scikit-learn estimator (assumed to accept an array input)
- For models whose features are numeric wavelengths, the spectrum is interpolated
  to those wavelengths. If the original spectrum contains an exact 2500.0 nm
  measurement, that value is preserved at the end.
- Writes one CSV per spectrum with predictions and a master summary CSV.
"""

import argparse
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import joblib
import pickle
import math
import warnings
from typing import Tuple

# ---------------------
# Utilities: reading spectra
# ---------------------
def read_with_specdal(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Try reading a spectrum using specdal.Spectrum.
    Returns (wavelengths, reflectance) or raises Exception.
    """
    try:
        import specdal
        from specdal.spectrum import Spectrum
    except Exception as e:
        raise ImportError(f"specdal unavailable or import failed: {e}")

    # Use constructor form that works in modern specdal: Spectrum(strpath) or fallback to read()
    try:
        spec = Spectrum(str(path))
    except TypeError:
        spec = Spectrum()
        spec.read(str(path))
    # Preferred measurement attribute is a pandas.Series
    ser = None
    if hasattr(spec, "measurement") and getattr(spec, "measurement") is not None:
        ser = spec.measurement
    elif hasattr(spec, "data") and getattr(spec, "data") is not None:
        ser = spec.data
    if ser is None:
        # fallback: some Spectrum implementations might hold arrays
        if hasattr(spec, "wavelengths") and hasattr(spec, "values"):
            return np.asarray(spec.wavelengths, float), np.asarray(spec.values, float)
        raise RuntimeError("specdal read succeeded but no measurement found on Spectrum object.")
    # Ensure numeric index and values
    wvl = np.asarray(ser.index.astype(float))
    refl = np.asarray(ser.values.astype(float))
    return wvl, refl

def fallback_text_parse(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Parse many common ASCII-exported ASD variants into (wavelengths, reflectance).
    Strategy: extract numeric tokens per line, keep rows with consistent token counts.
    """
    float_re = re.compile(r'[+-]?(?:\d+\.\d*|\d*\.\d+|\d+)(?:[eE][+-]?\d+)?')
    rows = []
    with open(path, 'r', errors='ignore') as f:
        for ln in f:
            if not ln.strip():
                continue
            toks = float_re.findall(ln)
            if toks:
                rows.append([float(t) for t in toks])
    if not rows:
        raise RuntimeError(f"No numeric data found in {path}")
    # pick the most common row length
    lengths = [len(r) for r in rows]
    common = max(set(lengths), key=lengths.count)
    rows = [r for r in rows if len(r) == common]
    arr = np.array(rows, dtype=float)
    if arr.shape[1] >= 2:
        # first col wavelength, second col reflectance
        return arr[:,0], arr[:,1]
    else:
        # single column: treat as reflectance, set wavelength to indices
        return np.arange(arr.shape[0], dtype=float), arr[:,0]

def read_spectrum(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """Try specdal, then fallback parser. Returns (wvl, refl)."""
    try:
        return read_with_specdal(path)
    except Exception:
        return fallback_text_parse(path)

# ---------------------
# Utilities: model loading & input preparation
# ---------------------
def load_pickle(path: Path):
    try:
        return joblib.load(path)
    except Exception:
        with open(path, 'rb') as f:
            return pickle.load(f)

def is_index_like(wvl: np.ndarray) -> bool:
    """Heuristic: if wavelengths are small sequential integers, treat as index-like."""
    return wvl.max() <= 100 and (np.allclose(wvl, np.arange(len(wvl))) or np.all(np.diff(wvl) >= 0))

def interpolate_preserve_2500(orig_wvl: np.ndarray, refl: np.ndarray, target_wvl: np.ndarray) -> np.ndarray:
    """
    Interpolate reflectance to target_wvl.
    If the original contains exactly 2500 nm (within tol), preserve that value at the end.
    Use constant extension at edges (left/right).
    """
    refl_interp = np.interp(target_wvl, orig_wvl, refl, left=refl[0], right=refl[-1])
    idx_2500 = np.where(np.isclose(orig_wvl, 2500.0, atol=1e-3))[0]
    if len(idx_2500) > 0:
        refl_interp[-1] = float(refl[idx_2500[0]])
    else:
        # if orig max equals 2500 within tol
        if np.isclose(orig_wvl.max(), 2500.0, atol=1e-3):
            refl_interp[-1] = float(refl[orig_wvl.argmax()])
    return refl_interp

def prepare_input_for_model(wvl: np.ndarray, refl: np.ndarray, model_obj: object, model_info: dict):
    """
    Prepare a 1xN input array for model_obj:
    - If model_obj has a 'pipeline' key (and is a pipeline), try pipeline.predict with suitable shape.
    - If model_info contains 'features' that are numeric wavelengths, interpolate spectrum to those wavelengths.
    - Otherwise try length matching and, if necessary, resample linearly to match model input length.
    Returns an (1, n_features) numpy array ready for model.predict()
    """
    # If model_obj is a dict that contains 'pipeline' or 'model' or 'features'
    # model_info may include parsed 'feature_wvl' etc.
    # Prefer exact feature wavelength matching
    if isinstance(model_obj, dict):
        # pipeline present?
        if 'pipeline' in model_obj and model_obj['pipeline'] is not None:
            pipeline = model_obj['pipeline']
            # pipeline.n_features_in_ could tell required input length
            req = getattr(pipeline, "n_features_in_", None)
            if req is None:
                # pipeline likely expects LVs (we should provide LVs), caller handles when needed
                raise RuntimeError("Pipeline found but required input length unknown.")
            # If pipeline expects LVs, we should not call it here to build X for it.
            # Return None to indicate caller must compute LVs (e.g., using a PLS model in the same pickle).
            return None
        # model + features?
        if 'model' in model_obj and 'features' in model_obj:
            features = model_obj['features']
            # try parse numeric feature names into wavelengths
            try:
                feat_wvl = np.array([float(f) for f in features])
                # interpolate to feat_wvl (preserve 2500 if present)
                # map index-like to real wavelengths 350..2500 if needed
                if is_index_like(wvl):
                    orig_wvl = np.linspace(350.0, 2500.0, len(wvl))
                else:
                    orig_wvl = wvl
                refl_interp = interpolate_preserve_2500(orig_wvl, refl, feat_wvl)
                return refl_interp.reshape(1, -1)
            except Exception:
                # features are not numeric; maybe they are names; if count matches, use direct mapping
                if len(features) == len(refl):
                    return refl.reshape(1, -1)
                else:
                    raise RuntimeError("Model features non-numeric and length mismatch with spectrum.")
    # If model_obj is a plain estimator
    if hasattr(model_obj, "n_features_in_"):
        req = int(model_obj.n_features_in_)
        # if lengths match
        if req == len(refl):
            return refl.reshape(1, -1)
        else:
            # interpolate based on index -> assume training grid 350..2500
            if is_index_like(wvl):
                orig_wvl = np.linspace(350.0, 2500.0, len(wvl))
            else:
                orig_wvl = wvl
            target_wvl = np.linspace(orig_wvl.min(), orig_wvl.max(), req)
            refl_interp = np.interp(target_wvl, orig_wvl, refl, left=refl[0], right=refl[-1])
            # preserve 2500 if present
            if np.isclose(orig_wvl.max(), 2500.0, atol=1e-3):
                refl_interp[-1] = float(refl[orig_wvl.argmax()])
            return refl_interp.reshape(1, -1)
    # Last resort: return raw reflectance as 1xN
    return refl.reshape(1, -1)

# ---------------------
# Main batch loop
# ---------------------
def batch_process(input_dir: Path, model_dir: Path, output_dir: Path, file_glob="*.asd", overwrite=False):
    input_dir = Path(input_dir)
    model_dir = Path(model_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load all model pickles
    pkl_list = sorted(model_dir.glob("*.pkl"))
    if not pkl_list:
        raise RuntimeError(f"No .pkl models found in {model_dir}")
    models = {}
    for p in pkl_list:
        try:
            obj = load_pickle(p)
        except Exception as e:
            warnings.warn(f"Failed to load {p}: {e}")
            continue
        models[p.name] = obj

    # Prepare summary table
    summary_rows = []

    # Find input files (supports many extensions)
    ext_list = ["*.asd", "*.sed", "*.sig", "*.txt", "*.csv"]
    files = []
    for ext in ext_list:
        files.extend(sorted(input_dir.glob(ext)))
    if not files:
        raise RuntimeError(f"No spectrum files found in {input_dir} for globs {ext_list}")

    for file_path in files:
        print(f"\nProcessing {file_path.name} ...")
        try:
            wvl, refl = read_spectrum(file_path)
        except Exception as e:
            warnings.warn(f"Failed to read {file_path}: {e}")
            continue

        # map index-like wavelengths to real axis 350..2500 if needed
        if is_index_like(wvl):
            orig_wvl = np.linspace(350.0, 2500.0, len(wvl))
        else:
            orig_wvl = wvl.astype(float)

        # per-file predictions container
        file_preds = {"file": file_path.name}

        # for each model, attempt to create input and predict
        for mname, mobj in models.items():
            try:
                # If this model dict contains 'pls_model' and 'pipeline' pattern, handle accordingly (common in your files)
                if isinstance(mobj, dict) and 'pls_model' in mobj and 'pipeline' in mobj:
                    # Interpolate raw reflectance to training grid 350..2500 step 5nm (431 points) and preserve 2500
                    target_wvl = np.arange(350.0, 2500.0 + 1e-6, 5.0)
                    refl_interp = np.interp(target_wvl, orig_wvl, refl, left=refl[0], right=refl[-1])
                    # preserve 2500 if present
                    idx_2500 = np.where(np.isclose(orig_wvl, 2500.0, atol=1e-3))[0]
                    if len(idx_2500) > 0:
                        refl_interp[-1] = float(refl[idx_2500[0]])
                    # Optionally apply Savitzky-Golay derivative if provided by model_info (many PLS cases)
                    # If the pickle contains instructions, follow them; otherwise not apply automatically
                    if mobj.get('apply_savgol', False):
                        from scipy.signal import savgol_filter
                        X = refl_interp.reshape(1, -1)
                        X = savgol_filter(X, 11, 2, deriv=1, axis=1)
                    else:
                        X = refl_interp.reshape(1, -1)
                    # Transform with PLS then predict with pipeline
                    pls = mobj['pls_model']
                    pipeline = mobj['pipeline']
                    lvs = pls.transform(X)
                    pred = pipeline.predict(lvs)
                    pred_val = float(np.asarray(pred).ravel()[0])
                    file_preds[mname] = pred_val
                    # save LVs per model if desired
                    file_preds[f"{mname}_lvs"] = lvs.ravel().tolist()
                else:
                    # Generic prepare path
                    prepared = prepare_input_for_model(orig_wvl, refl, mobj, {})
                    if prepared is None:
                        # the model likely expects LVs or special input; try heuristics:
                        # 1) if dict contains 'pls_model', compute LVs with it
                        if isinstance(mobj, dict) and 'pls_model' in mobj:
                            pls = mobj['pls_model']
                            # ensure reflectance interpolated to pls.n_features_in_
                            n_req = int(getattr(pls, "n_features_in_", pls.x_loadings_.shape[0] if hasattr(pls, "x_loadings_") else len(refl)))
                            target_wvl = np.linspace(orig_wvl.min(), orig_wvl.max(), n_req)
                            refl_interp = np.interp(target_wvl, orig_wvl, refl, left=refl[0], right=refl[-1])
                            X = refl_interp.reshape(1, -1)
                            lvs = pls.transform(X)
                            # if dict has pipeline, use it
                            if 'pipeline' in mobj and mobj['pipeline'] is not None:
                                pred = mobj['pipeline'].predict(lvs)
                                file_preds[mname] = float(pred.ravel()[0])
                            else:
                                file_preds[mname] = float(lvs.ravel()[0])  # fallback
                            file_preds[f"{mname}_lvs"] = lvs.ravel().tolist()
                        else:
                            raise RuntimeError("Model needs special handling; no prepareable features detected.")
                    else:
                        # prepared is 1xN array
                        # If dict with 'model' key, use it; else if plain estimator use it directly
                        estimator = None
                        if isinstance(mobj, dict) and 'model' in mobj:
                            estimator = mobj['model']
                        elif hasattr(mobj, "predict"):
                            estimator = mobj
                        else:
                            raise RuntimeError("No usable estimator found in model object.")
                        # If the estimator expects a DataFrame with named columns and we have 'features' metadata, use that
                        if isinstance(mobj, dict) and 'features' in mobj:
                            cols = mobj['features']
                            # if columns are numeric, convert to strings for DataFrame
                            df = pd.DataFrame(prepared, columns=[str(c) for c in cols])
                            pred = estimator.predict(df)
                        else:
                            pred = estimator.predict(prepared)
                        file_preds[mname] = float(np.asarray(pred).ravel()[0])
            except Exception as e:
                file_preds[mname] = None
                file_preds[f"{mname}_error"] = str(e)
                warnings.warn(f"Model {mname} failed on {file_path.name}: {e}")

        # Write per-file results
        out_csv = output_dir / f"{file_path.stem}_predictions.csv"
        pd.DataFrame([file_preds]).to_csv(out_csv, index=False)
        print(f"Saved predictions to {out_csv}")

        # Add to master summary (flatten numeric-only columns)
        summary_row = {"file": file_path.name}
        for k,v in file_preds.items():
            if k == "file": continue
            # only add scalar numeric preds to master summary; leave LV lists out
            if isinstance(v, (int, float, np.floating, np.integer)) and not (pd.isna(v)):
                summary_row[k] = v
            elif v is None:
                summary_row[k] = ""
        summary_rows.append(summary_row)

    # Save master summary
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = output_dir / "batch_predictions_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"\nWrote summary file: {summary_csv}")

# ---------------------
# CLI
# ---------------------
def main():
    parser = argparse.ArgumentParser(description="Batch predict ASD spectra with no GUI.")
    parser.add_argument("--input-dir", required=True, help="Directory containing ASD (and other) spectral files.")
    parser.add_argument("--model-dir", required=True, help="Directory containing .pkl model artifacts.")
    parser.add_argument("--output-dir", required=True, help="Directory to write per-file and summary CSVs.")
    parser.add_argument("--overwrite", action="store_true", help="Overwrite existing outputs.")
    args = parser.parse_args()
    batch_process(Path(args.input_dir), Path(args.model_dir), Path(args.output_dir), overwrite=args.overwrite)

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
#rejith code with batch and resampling
"""
Batch predict: always resample to 350..2500 nm @ 5 nm (431 points) and apply
Savitzky-Golay (1st derivative) before passing inputs to models.

Usage:
    python batch_predict_5nm_sg.py --input-dir ./asds --model-dir ./models --output-dir ./out
"""
from pathlib import Path
import argparse
import re
import json
import numpy as np
import pandas as pd
import joblib, pickle, warnings
from scipy.signal import savgol_filter

# -----------------------
# Config
# -----------------------
TARGET_WVL = np.arange(350.0, 2500.0 + 1e-6, 5.0)  # 431 points
SG_WINDOW = 11   # must be odd and <= n_features
SG_POLY = 2
SG_DERIV = 1     # first derivative

# -----------------------
# Helpers
# -----------------------
def load_pickle(path: Path):
    try:
        return joblib.load(path)
    except Exception:
        with open(path, "rb") as f:
            return pickle.load(f)

def is_index_like(wvl: np.ndarray) -> bool:
    return wvl.max() <= 100 and (np.allclose(wvl, np.arange(len(wvl))) or np.all(np.diff(wvl) >= 0))

def fallback_text_parse(path: Path):
    float_re = re.compile(r'[+-]?(?:\d+\.\d*|\d*\.\d+|\d+)(?:[eE][+-]?\d+)?')
    rows = []
    with open(path, 'r', errors='ignore') as f:
        for ln in f:
            if not ln.strip(): continue
            toks = float_re.findall(ln)
            if toks: rows.append([float(t) for t in toks])
    if not rows:
        raise RuntimeError(f"No numeric data found in {path}")
    lengths = [len(r) for r in rows]
    common = max(set(lengths), key=lengths.count)
    rows = [r for r in rows if len(r) == common]
    arr = np.array(rows, dtype=float)
    if arr.shape[1] >= 2:
        return arr[:,0], arr[:,1]
    else:
        return np.arange(arr.shape[0], dtype=float), arr[:,0]

def try_specdal_read(path: Path):
    """
    Try reading with specdal.Spectrum if available. If not available or fails,
    raise exception (caller should fallback to text parser).
    """
    try:
        import specdal
        from specdal.spectrum import Spectrum
    except Exception as e:
        raise ImportError("specdal not available: " + str(e))
    try:
        spec = Spectrum(str(path))
    except TypeError:
        spec = Spectrum()
        spec.read(str(path))
    ser = getattr(spec, "measurement", None) or getattr(spec, "data", None)
    if ser is None:
        # no pandas Series found
        if hasattr(spec, "wavelengths") and hasattr(spec, "values"):
            return np.asarray(spec.wavelengths, float), np.asarray(spec.values, float)
        raise RuntimeError("specdal read succeeded but no measurement found")
    return np.asarray(ser.index.astype(float)), np.asarray(ser.values.astype(float))

def read_spectrum(path: Path):
    # Try specdal first, else fallback
    try:
        return try_specdal_read(path)
    except Exception:
        return fallback_text_parse(path)

def interpolate_preserve2500(orig_wvl, refl, target_wvl=TARGET_WVL):
    """Interpolate from orig_wvl->target_wvl, preserve 2500nm if present in orig_wvl."""
    refl_interp = np.interp(target_wvl, orig_wvl, refl, left=refl[0], right=refl[-1])
    idx_2500 = np.where(np.isclose(orig_wvl, 2500.0, atol=1e-3))[0]
    if len(idx_2500) > 0:
        refl_interp[-1] = float(refl[idx_2500[0]])
    else:
        # if orig max equals 2500 within tol, use that
        if np.isclose(orig_wvl.max(), 2500.0, atol=1e-3):
            refl_interp[-1] = float(refl[orig_wvl.argmax()])
    return refl_interp

def apply_savgol_first_derivative(refl_1xN, window=SG_WINDOW, poly=SG_POLY, deriv=SG_DERIV):
    # refl_1xN shape (1,N)
    N = refl_1xN.shape[1]
    if window >= N:
        # if window too large, reduce to nearest odd < N
        w = N-1 if (N-1) % 2 == 1 else N-2
        if w < 3:
            w = 3
        window = int(w)
    # ensure window odd
    if window % 2 == 0:
        window -= 1
    return savgol_filter(refl_1xN, window_length=window, polyorder=poly, deriv=deriv, axis=1)

# -----------------------
# Main batch logic
# -----------------------
def batch_predict(input_dir, model_dir, out_dir):
    input_dir = Path(input_dir)
    model_dir = Path(model_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # load models
    pkl_files = sorted(model_dir.glob("*.pkl"))
    if not pkl_files:
        raise RuntimeError("No .pkl models found in model_dir")
    models = {}
    for p in pkl_files:
        try:
            models[p.name] = load_pickle(p)
        except Exception as e:
            warnings.warn(f"Cannot load {p.name}: {e}")

    # collect input files (common extensions)
    files = []
    for ext in ("*.asd","*.sed","*.sig","*.txt","*.csv"):
        files += sorted(input_dir.glob(ext))
    if not files:
        raise RuntimeError("No spectral files found in input_dir")

    summary = []
    for file in files:
        print(f"Processing {file.name} ...")
        try:
            wvl, refl = read_spectrum(file)
        except Exception as e:
            warnings.warn(f"Read failed for {file.name}: {e}")
            continue

        # map index-like to 350..2500
        if is_index_like(wvl):
            orig_wvl = np.linspace(350.0, 2500.0, len(wvl))
        else:
            orig_wvl = wvl.astype(float)

        # resample to target 5nm grid and preserve 2500 if present
        refl_431 = interpolate_preserve2500(orig_wvl, refl, TARGET_WVL)
        # shape to (1,431)
        X = refl_431.reshape(1, -1)

        # apply SG derivative (1st derivative) on the 431-band vector
        X_sg = apply_savgol_first_derivative(X, window=SG_WINDOW, poly=SG_POLY, deriv=SG_DERIV)

        # Per-model predictions
        preds = {"file": file.name}
        for mname, mobj in models.items():
            try:
                # common pattern in your pickles: dict with 'pls_model' + 'pipeline'
                if isinstance(mobj, dict) and 'pls_model' in mobj and 'pipeline' in mobj:
                    pls = mobj['pls_model']
                    pipeline = mobj['pipeline']
                    # Use SG-processed reflectance as input to PLS.transform
                    lvs = pls.transform(X_sg)
                    pred = pipeline.predict(lvs)
                    preds[mname] = float(pred.ravel()[0])
                    preds[f"{mname}_lvs"] = lvs.ravel().tolist()
                else:
                    # Generic estimator or dict{'model','features'}: feed SG reflectance if lengths match,
                    # otherwise try interpolation to model expected shape (but main goal is 5nm+SG)
                    estimator = None
                    if isinstance(mobj, dict) and 'model' in mobj:
                        estimator = mobj['model']
                        features = mobj.get('features', None)
                        if features:
                            # if features are numeric wavelengths and length == 431, use X_sg
                            try:
                                feat_w = np.array([float(f) for f in features])
                                if len(feat_w) == len(TARGET_WVL):
                                    # use SG reflectance directly (columns correspond to TARGET_WVL)
                                    df = pd.DataFrame(X_sg, columns=[str(w) for w in TARGET_WVL])
                                    pred = estimator.predict(df)
                                    preds[mname] = float(pred.ravel()[0])
                                else:
                                    # interpolate SG reflectance to feature wavelengths
                                    refl_for_feat = np.interp(feat_w, TARGET_WVL, X_sg.ravel())
                                    df = pd.DataFrame(refl_for_feat.reshape(1,-1), columns=[str(f) for f in features])
                                    pred = estimator.predict(df)
                                    preds[mname] = float(pred.ravel()[0])
                            except Exception:
                                # fallback: try raw estimator predict on SG vector if size matches
                                if hasattr(estimator, "n_features_in_") and estimator.n_features_in_ == X_sg.shape[1]:
                                    pred = estimator.predict(X_sg)
                                    preds[mname] = float(pred.ravel()[0])
                                else:
                                    # attempt to resample SG to estimator.n_features_in_ if available
                                    if hasattr(estimator, "n_features_in_"):
                                        req = int(estimator.n_features_in_)
                                        target_idx = np.linspace(0, X_sg.shape[1]-1, req)
                                        refl_resampled = np.interp(target_idx, np.arange(X_sg.shape[1]), X_sg.ravel())
                                        pred = estimator.predict(refl_resampled.reshape(1,-1))
                                        preds[mname] = float(pred.ravel()[0])
                                    else:
                                        raise RuntimeError("Cannot prepare input for model")
                    elif hasattr(mobj, "predict"):
                        estimator = mobj
                        # if estimator expects 431 features, pass SG; else resample to match
                        if hasattr(estimator, "n_features_in_") and estimator.n_features_in_ == X_sg.shape[1]:
                            pred = estimator.predict(X_sg)
                            preds[mname] = float(pred.ravel()[0])
                        elif hasattr(estimator, "n_features_in_"):
                            req = int(estimator.n_features_in_)
                            idx = np.linspace(0, X_sg.shape[1]-1, req)
                            refl_resampled = np.interp(idx, np.arange(X_sg.shape[1]), X_sg.ravel())
                            pred = estimator.predict(refl_resampled.reshape(1,-1))
                            preds[mname] = float(pred.ravel()[0])
                        else:
                            # no info: try directly
                            pred = estimator.predict(X_sg)
                            preds[mname] = float(pred.ravel()[0])
                    else:
                        raise RuntimeError("Unsupported model object structure")
            except Exception as e:
                preds[mname] = None
                preds[f"{mname}_error"] = str(e)
                warnings.warn(f"Model {mname} failed for {file.name}: {e}")

        # save per-file CSV
        out_csv = out_dir / f"{file.stem}_predictions.csv"
        pd.DataFrame([preds]).to_csv(out_csv, index=False)

        # save SG LVs if any of models produced them (collect last one)
        # (we prioritized PLS models and saved their LVs into preds keys if present)
        lv_keys = [k for k in preds.keys() if k.endswith("_lvs")]
        if lv_keys:
            # take first lv key
            lvvals = preds[lv_keys[0]]
            with open(out_dir / f"{file.stem}_lvs.json", "w") as f:
                json.dump({"latent_variables": lvvals}, f, indent=2)

        # append to summary
        summary_row = {"file": file.name}
        for k, v in preds.items():
            if k == "file": continue
            if isinstance(v, (int, float, np.floating, np.integer)) and not pd.isna(v):
                summary_row[k] = v
            else:
                summary_row[k] = "" if v is None else v
        summary.append(summary_row)

        print(f"Saved: {out_csv}")

    # write master summary
    summary_df = pd.DataFrame(summary)
    summary_csv = out_dir / "batch_predictions_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    print("Wrote summary:", summary_csv)

# -----------------------
# CLI
# -----------------------
def main():
    parser = argparse.ArgumentParser(description="Batch predict 5nm + SG")
    parser.add_argument("--input-dir", required=True)
    parser.add_argument("--model-dir", required=True)
    parser.add_argument("--output-dir", required=True)
    args = parser.parse_args()
    batch_predict(args.input_dir, args.model_dir, args.output_dir)

if __name__ == "__main__":
    main()


In [2]:
import pickle


with open('D:\\Tarun\\vasudha_python\\MIR_N_SVM.pkl', 'rb') as f:
    data = pickle.load(f)

AttributeError: Can't get attribute 'PLSFeatureExtractor' on <module '__main__'>

In [8]:
from pathlib import Path

# Define the path to the target folder
folder_path = Path('C:\\Users\\DRONE LAB\\Documents\\vasudha') # Replace with your actual path

print(f"--- Contents of {folder_path} ---")

# Use rglob('*') to find all files and directories recursively
for path_object in folder_path.rglob('*'):
    if path_object.is_file():
        print(f"File: {path_object}")
    elif path_object.is_dir():
        print(f"Folder: {path_object}")

--- Contents of C:\Users\DRONE LAB\Documents\vasudha ---
Folder: C:\Users\DRONE LAB\Documents\vasudha\Alluvial
Folder: C:\Users\DRONE LAB\Documents\vasudha\Black
Folder: C:\Users\DRONE LAB\Documents\vasudha\common
Folder: C:\Users\DRONE LAB\Documents\vasudha\Desert
Folder: C:\Users\DRONE LAB\Documents\vasudha\Grey and Brown
Folder: C:\Users\DRONE LAB\Documents\vasudha\Laterite
Folder: C:\Users\DRONE LAB\Documents\vasudha\Mixed red and black
Folder: C:\Users\DRONE LAB\Documents\vasudha\Mountain
Folder: C:\Users\DRONE LAB\Documents\vasudha\Red
Folder: C:\Users\DRONE LAB\Documents\vasudha\Alluvial\mir
Folder: C:\Users\DRONE LAB\Documents\vasudha\Alluvial\vnir
Folder: C:\Users\DRONE LAB\Documents\vasudha\Black\mir
Folder: C:\Users\DRONE LAB\Documents\vasudha\Black\vnir
Folder: C:\Users\DRONE LAB\Documents\vasudha\common\mir
Folder: C:\Users\DRONE LAB\Documents\vasudha\common\vnir
Folder: C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR
Folder: C:\Users\DRONE LAB\Documents\vasudha\comm

In [6]:
#!/usr/bin/env python3
"""
rename_models_to_canonical.py

Dry-run by default: prints proposed renames.
Run with --apply to perform changes.

Usage:
    python rename_models_to_canonical.py         # dry-run
    python rename_models_to_canonical.py --apply # actually rename
"""

import re
import argparse
from pathlib import Path

# adjust base to your vasudha folder
BASE = Path(r"C:\Users\DRONE LAB\Documents\vasudha")

LABEL_TOKENS = {
    'n': 'n', 'nitrogen': 'n',
    'p': 'p', 'phosphorus': 'p',
    'k': 'k', 'potassium': 'k',
    'oc': 'oc', 'organiccarbon': 'oc', 'organic_carbon': 'oc', 'organic': 'oc',
    'ph': 'ph', 'pH': 'ph', 'ph_value': 'ph',
    'ec': 'ec', 'conductivity': 'ec'
}

# mapping of long component names -> canonical short component key
COMPONENT_REPLACEMENTS = {
    r'partial_least_squares_regression_model': 'primary_pls_model',
    r'partial_least_squares_regression': 'plsr',
    r'primary_pls_model': 'primary_pls_model',
    r'primary_pls_scaler': 'primary_pls_scaler',
    r'secondary_plsr_model': 'secondary_plsr_model',
    r'secondary_pls_model': 'secondary_plsr_model',
    r'primary_pls': 'primary_pls_model',
    r'random_forest_regressor_model': 'secondary_rf_model',
    r'random_forest_regressor': 'rfr_model',
    r'rfr': 'rfr',
    r'_rfr_': 'rfr',
    r'secondary_rf_model': 'secondary_rf_model',
    r'secondary_svr_model': 'secondary_svr_model',
    r'svr': 'secondary_svr_model',
    r'svm': 'secondary_svr_model',
    r'boxcox_lambda': 'boxcox_lambda',
    r'boxcox_shift': 'boxcox_shift',
    r'primary_pls_scaler': 'primary_pls_scaler',
    r'secondary_lv_scaler': 'secondary_lv_scaler',
    r'lv_scaler': 'secondary_lv_scaler',
    r'pls': 'plsr',
    r'plsr': 'plsr',
    r'plsr_svm': 'plsr_svm',
    r'model': 'model',
    r'original': None,  # folder token only, skip
    # Add more rules if you find other token variants
}

SENSOR_TOKENS = {
    'asd': 'asd',
    'vnir': 'asd',
    'mir': 'mir',
    'mid_ir': 'mir',
    'mid-ir': 'mir',
    'midir': 'mir'
}

def detect_label_from_name(name):
    name_l = name.lower()
    for token, lab in LABEL_TOKENS.items():
        if re.search(rf'(?:^|_){re.escape(token)}(?:_|$)', name_l):
            return lab
    # fallback: check presence anywhere
    for token, lab in LABEL_TOKENS.items():
        if token in name_l:
            return lab
    return None

def detect_sensor_from_path(filepath):
    # look at path parts for 'vnir'/'mir'/'asd'
    p = Path(filepath).resolve()
    for part in p.parts:
        low = part.lower()
        if low in ('vnir','asd'):
            return 'asd'
        if low in ('mir','mid_ir','mid-ir','midir'):
            return 'mir'
    return None

def find_component_token(name):
    name_l = name.lower()
    for pat, repl in COMPONENT_REPLACEMENTS.items():
        if repl is None: 
            continue
        if re.search(rf'{pat}', name_l):
            return repl
    # heuristic: choose last meaningful token before sensor or extension
    # split by underscores and try to find a token that's not sensor/label
    tokens = re.split(r'[_\-\s]+', name_l)
    tokens = [t for t in tokens if t and t not in LABEL_TOKENS and t not in SENSOR_TOKENS and t != 'pkl' and t != 'original']
    if tokens:
        return tokens[0]
    return 'model'

def canonical_name_for_path(p: Path):
    name = p.name
    stem = p.stem
    lower_stem = stem.lower()

    # determine sensor: check filename then path
    sensor = None
    for tok in SENSOR_TOKENS:
        if re.search(rf'(?:^|_){re.escape(tok)}(?:_|$)', lower_stem):
            sensor = SENSOR_TOKENS[tok]
            break
    if sensor is None:
        sensor = detect_sensor_from_path(p) or ''

    # determine label
    label = detect_label_from_name(lower_stem)
    if label is None:
        # maybe label at the end like ...ModelN or ...ModelEC
        m = re.search(r'([npkoc]{1,2}|ec|ph)$', lower_stem)
        if m:
            label = m.group(1)
        else:
            # fallback to generic
            label = '__generic__'

    # determine component
    comp = find_component_token(lower_stem)

    # clean comp: replace spaces/dots with underscore
    comp = comp.replace('.', '_').replace(' ', '_')

    # build canonical
    parts = [label, comp]
    if sensor:
        parts.append(sensor)
    newname = '_'.join([p for p in parts if p]) + '.pkl'
    # normalize underscores
    newname = re.sub(r'__+', '_', newname)
    newname = newname.lower()
    return newname

def scan_and_propose(base_dir: Path, apply_changes=False, dry_run=True):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        print("Base path not found:", base_dir)
        return

    actions = []
    for fp in base_dir.rglob("*.pkl"):
        if fp.is_file():
            new_fn = canonical_name_for_path(fp)
            new_fp = fp.with_name(new_fn)
            if new_fp.resolve() == fp.resolve():
                # already canonical or name unchanged
                continue
            # avoid collisions: if target name exists, propose a safe name (append index)
            if new_fp.exists():
                i = 1
                candidate = new_fp.with_stem(new_fp.stem + f"_{i}")
                while candidate.exists():
                    i += 1
                    candidate = new_fp.with_stem(new_fp.stem + f"_{i}")
                new_fp = candidate

            actions.append((fp, new_fp))

    if not actions:
        print("No files to rename (all look canonical or nothing found).")
        return

    print(f"Found {len(actions)} proposed renames (dry_run={dry_run}, apply={apply_changes}):\n")
    for src, dst in actions:
        print(f"{src}  ->  {dst}")

    if apply_changes and not dry_run:
        for src, dst in actions:
            try:
                src.rename(dst)
            except Exception as e:
                print(f"Failed to rename {src} -> {dst}: {e}")
        print("Rename complete.")
    else:
        print("\nRun with --apply --no-dry-run to actually perform renames (use carefully).")

if __name__ == "__main__":
    import sys
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--apply", action="store_true", help="Actually perform renames")
    ap.add_argument("--no-dry-run", action="store_true", help="If set along with --apply, will perform actual renames")
    ap.add_argument("--base", default=str(BASE), help="Base vasudha models folder")

    # Use parse_known_args so Jupyter's extra args won't break us.
    args, unknown = ap.parse_known_args()
    if unknown:
        # Optional: log that unknown args were ignored (useful in notebooks)
        print("Ignored unknown args:", unknown, file=sys.stderr)

    apply_changes = args.apply and args.no_dry_run
    scan_and_propose(Path(args.base), apply_changes=apply_changes, dry_run=not apply_changes)



Ignored unknown args: ['-f', 'C:\\Users\\DRONE LAB\\AppData\\Roaming\\jupyter\\runtime\\kernel-23bc9c03-0cd5-47b0-8c3a-df9129406e76.json']


Found 203 proposed renames (dry_run=True, apply=False):

C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\boxcox_lambda_ec_mir.pkl  ->  C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\ec_boxcox_lambda_mir.pkl
C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\boxcox_lambda_k_mir.pkl  ->  C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\k_boxcox_lambda_mir.pkl
C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\boxcox_lambda_n_mir.pkl  ->  C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\n_boxcox_lambda_mir.pkl
C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\boxcox_lambda_oc_mir.pkl  ->  C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\oc_boxcox_lambda_mir.pkl
C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\boxcox_lambda_ph_mir.pkl  ->  C:\Users\DRONE LAB\Documents\vasudha\common\mir\HPLSR\original\ph_boxcox_lambda_mir.pkl
C:\Users\DRONE LAB\Documents\vasudha\commo